# MODE 2 — M2_F01 ANIMATION VALIDATOR — CONTROL

> Phase 8 — Dual Pipeline Doctrine — v1.0.0

**Rôle :** Vérification pré-vol et diagnostics de la frégate M2_F01.

| Loi | Règle |
|-----|-------|
| R-01 | Isolation — copie indépendante Mode 2 |
| R-02 | GLB obligatoire avec animations embarquées |
| R-03 | durée_audio <= durée_animation |

In [ ]:
# ── CELLULE 0 — CONFIGURATION ────────────────────────────────
from pathlib import Path
import json

FREGATE_ROOT = Path("../")  # 07_M2_F01_ANIMATION/
CODEBASE     = FREGATE_ROOT / "CODEBASE"
IN_GLB       = FREGATE_ROOT / "IN_GLB_AVATAR"
IN_AUDIO     = FREGATE_ROOT / "IN_AUDIO"
OUT_VALID    = FREGATE_ROOT / "OUT_VALIDATED"
OUT_REPORT   = FREGATE_ROOT / "OUT_REPORT"

print("=== M2_F01 CONTROL — PRÉ-VOL ===")
print(f"Frégate root : {FREGATE_ROOT.resolve()}")

In [ ]:
# ── CELLULE 1 — VÉRIFICATION STRUCTURE ───────────────────────
dirs = [IN_GLB, IN_AUDIO, OUT_VALID, OUT_REPORT]
for d in dirs:
    status = "✅" if d.exists() else "❌"
    print(f"  {status} {d.name}")

# Check dépendances Python
print("\n=== DÉPENDANCES ===")
for pkg in ["pygltflib", "librosa", "pydub", "soundfile"]:
    try:
        __import__(pkg)
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ⚠️  {pkg} (non installé — optionnel)")

In [ ]:
# ── CELLULE 2 — INSPECTION GLB ───────────────────────────────
glbs = list(IN_GLB.glob("*.glb"))
print(f"GLB trouvés : {len(glbs)}")
for g in glbs:
    size_mb = g.stat().st_size / (1024*1024)
    with open(g, "rb") as f:
        magic = f.read(4)
    valid_magic = "✅" if magic == b"glTF" else "❌"
    print(f"  {valid_magic} {g.name} ({size_mb:.2f} MB)")

In [ ]:
# ── CELLULE 3 — INSPECTION AUDIO ─────────────────────────────
audio_files = []
for ext in [".wav", ".mp3", ".ogg", ".aac", ".flac"]:
    audio_files.extend(IN_AUDIO.glob(f"*{ext}"))

print(f"Audio trouvés : {len(audio_files)}")
for a in audio_files:
    size_kb = a.stat().st_size / 1024
    print(f"  📢 {a.name} ({size_kb:.1f} KB)")

if not audio_files:
    print("  ℹ️  Aucun audio — mode sans audio (R-03 non applicable)")

In [ ]:
# ── CELLULE 4 — LECTURE RAPPORT (si existant) ─────────────────
report_path = OUT_REPORT / "m2_f01_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    print(f"Statut        : {report['status']}")
    print(f"Timestamp     : {report['timestamp']}")
    glb_val = report.get('glb_validation', {})
    print(f"GLB valide    : {glb_val.get('valid')}")
    print(f"Animations    : {len(glb_val.get('animations', []))}")
    print(f"Durée anim    : {glb_val.get('total_duration_s', 0):.3f}s")
    r03 = report.get('loi_r03', {})
    print(f"LOI R-03      : {r03.get('status')}")
    if report.get('errors'):
        print(f"Erreurs       : {report['errors']}")
else:
    print("Aucun rapport — lancer la production d'abord")